In [3]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient

In [4]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

### RAG Pipeline

In [5]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [6]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

In [7]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt


def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response

def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [8]:
output = rag_pipeline("Do you have any earphones?")

In [9]:
output

{'data_object': RAGGenerationResponse(answer='Yes—there are several earphone options available, including kids wired earphones with a carry case (Volkano), a 2-pack of wired iPhone-compatible earphones with microphone/remote, two models of wireless Bluetooth earbuds with charging case (up to 40H playtime, with waterproof ratings), and a Bluetooth 5.3 sports headband headset with ENC microphone.'),
 'answer': 'Yes—there are several earphone options available, including kids wired earphones with a carry case (Volkano), a 2-pack of wired iPhone-compatible earphones with microphone/remote, two models of wireless Bluetooth earbuds with charging case (up to 40H playtime, with waterproof ratings), and a Bluetooth 5.3 sports headband headset with ENC microphone.',
 'question': 'Do you have any earphones?',
 'retrieved_context_ids': ['B0BBF2VC6X',
  'B0CH8DRD6K',
  'B09X9838WY',
  'B0BRV544MV',
  'B0CFHWF326'],
 'retrieved_context': ['Volkano Ninja Kids Earphones for Boys with Carry Case and Ke

In [10]:
print(output["answer"])

Yes—there are several earphone options available, including kids wired earphones with a carry case (Volkano), a 2-pack of wired iPhone-compatible earphones with microphone/remote, two models of wireless Bluetooth earbuds with charging case (up to 40H playtime, with waterproof ratings), and a Bluetooth 5.3 sports headband headset with ENC microphone.


### RAG Pipeline with Grounding Context

In [11]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="ID of the item used to answer the question")
    description: str = Field(description="Description of the item used to answer the question")

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")

In [12]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- If you are describing multiple products, list them out as a list.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "references": answer.references,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer


In [13]:
output = rag_pipeline("Do you have any earphones?")

In [14]:
output

{'data_object': RAGGenerationResponse(answer='Yes—there are multiple earphone options available, including wired and wireless types.', references=[RAGUsedContext(id='B0BBF2VC6X', description='Volkano Ninja Kids wired earphones (3.5mm) with safe volume limit and carry case.'), RAGUsedContext(id='B0CH8DRD6K', description='2-pack iPhone wired headphones (3.5mm, Apple MFi certified) with microphone and volume control.'), RAGUsedContext(id='B09X9838WY', description='Jesebang Bluetooth 5.2 wireless earbuds with charging case, mic, and IP7 waterproof rating.'), RAGUsedContext(id='B0BRV544MV', description='Wekily Bluetooth 5.3 wireless earbuds with 40H playtime, 4-mic clear call, and IPX7 waterproof rating.'), RAGUsedContext(id='B0CFHWF326', description='MUSICOZY Bluetooth 5.3 sports headband headphones with built-in ENC mic and 20+ hours playtime.')]),
 'answer': 'Yes—there are multiple earphone options available, including wired and wireless types.',
 'references': [RAGUsedContext(id='B0BBF2

In [16]:
print(output["answer"])

Yes—there are multiple earphone options available, including wired and wireless types.


In [17]:
output = rag_pipeline("Do you have any earphones?", top_k=10)

In [18]:
output

{'data_object': RAGGenerationResponse(answer='Yes— we have several earphones/earbud options available:\n\n- Volkano Ninja Kids Earphones (wired, 85dB volume limit) with carry case and keyring (Red/Blue).\n- 2 Pack iPhone Wired Headphones (3.5mm, Apple MFi certified) with microphone and volume control.\n- Jesebang Wireless Earbuds (Bluetooth 5.2, touch control, charging case, IP7).\n- Wekily Bluetooth 5.3 Wireless Earbuds (40H playtime, 4-mic clear call, IPX7).\n- MUSICOZY Bluetooth 5.3 Headband Headphones (ENC mic, 20+ hours playtime).\n- pamu S29 Wireless Earbuds (Bluetooth 5.2, ANC/ENC, charging case, IPX4).\n- Kids Wireless/3.5mm Jack Headphones (adjustable, foldable, built-in microphone).\n- Bluet00th Sleep Headband Headphones (sleep headband with earbuds, 10–12H).\n- RUNAR RNR1 Wireless Neckband Running Headphones (Bluetooth 5.0, sweatproof/rainproof, TF/SD compatible).\n\nWhich type are you looking for (kids, wired vs wireless, or sports/sleep)?', references=[RAGUsedContext(id='B

In [19]:
print(output["answer"])

Yes— we have several earphones/earbud options available:

- Volkano Ninja Kids Earphones (wired, 85dB volume limit) with carry case and keyring (Red/Blue).
- 2 Pack iPhone Wired Headphones (3.5mm, Apple MFi certified) with microphone and volume control.
- Jesebang Wireless Earbuds (Bluetooth 5.2, touch control, charging case, IP7).
- Wekily Bluetooth 5.3 Wireless Earbuds (40H playtime, 4-mic clear call, IPX7).
- MUSICOZY Bluetooth 5.3 Headband Headphones (ENC mic, 20+ hours playtime).
- pamu S29 Wireless Earbuds (Bluetooth 5.2, ANC/ENC, charging case, IPX4).
- Kids Wireless/3.5mm Jack Headphones (adjustable, foldable, built-in microphone).
- Bluet00th Sleep Headband Headphones (sleep headband with earbuds, 10–12H).
- RUNAR RNR1 Wireless Neckband Running Headphones (Bluetooth 5.0, sweatproof/rainproof, TF/SD compatible).

Which type are you looking for (kids, wired vs wireless, or sports/sleep)?


In [20]:
output = rag_pipeline("Do you have any earphones?", top_k=15)

In [21]:
print(output["answer"])

Yes—here are the earphones/earbuds currently available:

- Volkano Ninja Kids Earphones for Boys (3.5mm wired) with carry case (red/blue)
- 2 Pack iPhone Wired Headphones (3.5mm) with microphone and volume control (Apple MFi certified)
- Jesebang Wireless Earbuds (Bluetooth 5.2) with charging case, mic, touch control, IP7 waterproof (white)
- Wekily Bluetooth 5.3 Wireless Earbuds with charging case, 4-mic clear call, LED display, IPX7 waterproof
- MUSICOZY Bluetooth 5.3 Headband Headphones with ENC mic (sports/workouts)
- pamu Wireless Earbuds with ANC/ENC, Bluetooth 5.2, charging case, IPX4 (black)
- Kids Bluetooth/3.5mm Jack Headphones with volume control, foldable (yellow)
- Bluetooth Sleep Headband Earphones (sleep mask/headband style)


In [19]:
output

{'data_object': RAGGenerationResponse(answer='Yes—we have several earphones/earbuds available, including:\n\n- Volkano Ninja Kids Earphones (3.5mm wired) with carry case and keyring (Red/Blue)\n- 2 Pack iPhone Wired Headphones (Apple MFi certified, 3.5mm) with microphone and volume control\n- Jesebang Bluetooth 5.2 Wireless Earbuds (with charging case, IP7)\n- Wekily Bluetooth 5.3 Wireless Earbuds (with charging case, IPX7, 4-mic clear call)\n- MUSICOZY Bluetooth 5.3 Sports Headband (ENC mic, 20+ hours)\n- pamu Pamu Wireless Earbuds with ANC/ENC (Bluetooth 5.2, charging case)\n- Kids Wireless/3.5mm Jack Over-Ear Headphones (Yellow, with microphone, foldable)\n- Bluetooth Sleep Headband Earphones (sleep mask/headband, 10–12H)\n- RUNAR RNR1 Wireless Neckband Running Headphones (Bluetooth 5.0)\n', references=[RAGUsedContext(id='B0BBF2VC6X', description='Volkano Ninja Kids Earphones (3.5mm wired) with carry case and keyring; safe volume 85dB.'), RAGUsedContext(id='B0CH8DRD6K', description=